# YOLO-Fastest 人物検出モデル学習 (darknet版) - Ethos-U55 NPU対応

darknetフレームワークでYOLO-Fastest V1を学習し、TFLite INT8に変換する。

## Nota-NetsPresso版ノートブック (train_yolo_fastest_person_colab.ipynb) との違い

| 項目 | Nota-NetsPresso版 | **本ノートブック (darknet版)** |
|---|---|---|
| フレームワーク | PyTorch (YOLOv5ベース再実装) | **C (darknet原著)** |
| リポジトリ | Nota-NetsPresso/ModelZoo-YOLOFastest | **dog-qiuqiu/Yolo-Fastest** |
| TFLite変換 | ONNX -> onnx2tf -> TFLite | **darknet .weights -> Keras .h5 -> TFLite** |
| Ethos-U55実績 | SiLU活性化が未対応の可能性 | **顔検出サンプルと同一アーキテクチャ** |
| 出力構造 | YOLOv5形式 (単一テンソル可能性) | **2ブランチ (6x6+12x12), 顔検出と同一構造** |

## 背景 (Issue #128)
- YOLOv8系モデルはStridedSliceオペレータがEthos-U55未対応で3サブグラフに分割される
- Nota-NetsPresso版はonnx2tf変換後にVela AssertionErrorが発生
- 顔検出サンプル (emza-vs YOLO-Fastest V1, darknet由来) はEthos-U55で単一サブグラフ動作実績あり
- **本ノートブックは顔検出と同一の変換パス (darknet -> Keras -> TFLite INT8) を使用**

## モデル構成
- ベース: dog-qiuqiu/Yolo-Fastest (YOLO-Fastest V1)
- 入力: 192x192x3 (RGB, INT8)
- 出力: 2ブランチ (6x6 stride-32 + 12x12 stride-16), 各3アンカー x 6値 (x,y,w,h,obj,cls)
- クラス: 1 (person)
- 検出方式: アンカーベース (YOLOv3スタイル)

## 前提条件
- Google Colab (GPU: T4推奨)
- Google Driveに `fall_detection_dataset.zip` をアップロード済み

## ワークフロー
1. GPU確認・環境構築 (darknetコンパイル)
2. データセット準備 (darknet形式)
3. YOLO-Fastest cfgファイル作成
4. アンカー計算
5. 事前学習済み重み取得
6. モデル学習
7. 精度評価
8. darknet weights -> TFLite INT8 変換
9. モデル詳細確認
10. Vela互換性確認
11. 成果物ダウンロード

---
## Step 1: GPU確認・環境構築

**重要:** メニューの「ランタイム > ランタイムのタイプを変更」で **GPU (T4)** を選択してください。

darknetをGPU対応でコンパイルします。

In [ ]:
# GPU 確認
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print('=== GPU が利用可能です ===')
else:
    print('WARNING: GPU が検出されません。ランタイムを GPU に変更してください。')

In [ ]:
# Yolo-Fastest リポジトリをクローンしてdarknetをコンパイル
import os

DARKNET_DIR = '/content/Yolo-Fastest'

if not os.path.isdir(DARKNET_DIR):
    !git clone https://github.com/dog-qiuqiu/Yolo-Fastest.git {DARKNET_DIR}
else:
    print(f'{DARKNET_DIR} は既に存在します')

# darknetサブディレクトリの確認
# Yolo-Fastestリポジトリはdarknetのフォークを含む
darknet_makefile = os.path.join(DARKNET_DIR, 'Makefile')
if os.path.exists(darknet_makefile):
    print('Makefile が見つかりました')
else:
    # リポジトリ構造確認
    !ls -la {DARKNET_DIR}/
    print('\nWARNING: Makefileが見つかりません。リポジトリ構造を確認してください。')

In [ ]:
%%bash
# darknetをGPU対応でコンパイル
cd /content/Yolo-Fastest

# MakefileをGPU対応に修正
sed -i 's/GPU=0/GPU=1/' Makefile
sed -i 's/CUDNN=0/CUDNN=1/' Makefile
# OpenCVは学習に不要。Colab環境ではopencv-devが未インストールのため無効化
# デフォルトがOPENCV=1の場合もあるので、明示的に0に設定
sed -i 's/OPENCV=1/OPENCV=0/' Makefile

# コンパイル
make clean 2>/dev/null || true
make -j$(nproc)

# 確認
if [ -f ./darknet ]; then
    echo ''
    echo '=== darknet コンパイル成功 ==='
    ls -la ./darknet
else
    echo 'ERROR: darknet コンパイル失敗'
fi


---
## Step 2: データセット準備

### 事前準備 (ローカルPCで実行)

```bash
cd mimamori-sense/dataset/merged
zip -r fall_detection_dataset.zip images/ labels/
```

作成した `fall_detection_dataset.zip` を Google Drive のマイドライブ直下にアップロードしてください。

### darknet形式について

darknetは以下のファイルを必要とします:
- `obj.data`: クラス数、パス等の設定
- `obj.names`: クラス名一覧
- `train.txt`: 学習画像の絶対パス一覧
- `valid.txt`: 検証画像の絶対パス一覧
- 各画像と同名の `.txt` ラベルファイル (YOLO形式: class_id x_center y_center w h)

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob

DARKNET_DIR = '/content/Yolo-Fastest'
DATASET_DIR = '/content/dataset'
DATASET_ZIP = '/content/drive/MyDrive/fall_detection_dataset.zip'
DATA_DIR = os.path.join(DARKNET_DIR, 'data', 'person')
BACKUP_DIR = os.path.join(DARKNET_DIR, 'backup')
GDRIVE_BACKUP = '/content/drive/MyDrive/yolo_fastest_darknet_person'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)
os.makedirs(GDRIVE_BACKUP, exist_ok=True)

# データセット展開
if not os.path.isdir(os.path.join(DATASET_DIR, 'images')):
    if os.path.isfile(DATASET_ZIP):
        print('データセット展開中...')
        !mkdir -p {DATASET_DIR} && unzip -q {DATASET_ZIP} -d {DATASET_DIR}
        print('展開完了')
    else:
        print(f'ERROR: {DATASET_ZIP} が見つかりません')
        print('Google Driveに fall_detection_dataset.zip をアップロードしてください')
else:
    print('データセットは展開済みです')

# 検証
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    if os.path.isdir(img_dir):
        img_count = len([f for f in os.listdir(img_dir)
                         if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        lbl_count = len([f for f in os.listdir(lbl_dir)
                         if f.endswith('.txt')]) if os.path.isdir(lbl_dir) else 0
        print(f'  {split}: images={img_count}, labels={lbl_count}')

In [ ]:
import os
import glob

DARKNET_DIR = '/content/Yolo-Fastest'
DATASET_DIR = '/content/dataset'
DATA_DIR = os.path.join(DARKNET_DIR, 'data', 'person')

print('=== darknet形式ファイル生成 ===')

# ----- obj.names -----
names_path = os.path.join(DATA_DIR, 'obj.names')
with open(names_path, 'w') as f:
    f.write('person\n')
print(f'作成: {names_path}')

# ----- darknet用のラベル配置 -----
# darknetは画像と同じディレクトリに.txtラベルを置く規約
# 既存データセットはimages/とlabels/が分離しているのでシンボリックリンクを作成
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    if not os.path.isdir(img_dir) or not os.path.isdir(lbl_dir):
        continue
    linked = 0
    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith('.txt'):
            continue
        src = os.path.join(lbl_dir, lbl_file)
        dst = os.path.join(img_dir, lbl_file)
        if not os.path.exists(dst):
            os.symlink(src, dst)
            linked += 1
    print(f'  {split}: {linked} ラベルファイルをリンク済み')

# ----- train.txt / valid.txt -----
for split, filename in [('train', 'train.txt'), ('val', 'valid.txt')]:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    if not os.path.isdir(img_dir):
        print(f'WARNING: {img_dir} が見つかりません')
        continue
    images = sorted(glob.glob(os.path.join(img_dir, '*.jpg')) +
                    glob.glob(os.path.join(img_dir, '*.png')) +
                    glob.glob(os.path.join(img_dir, '*.jpeg')))
    list_path = os.path.join(DATA_DIR, filename)
    with open(list_path, 'w') as f:
        for img_path in images:
            f.write(img_path + '\n')
    print(f'作成: {list_path} ({len(images)} 画像)')

# ----- obj.data -----
data_path = os.path.join(DATA_DIR, 'obj.data')
with open(data_path, 'w') as f:
    f.write(f'classes = 1\n')
    f.write(f'train = {os.path.join(DATA_DIR, "train.txt")}\n')
    f.write(f'valid = {os.path.join(DATA_DIR, "valid.txt")}\n')
    f.write(f'names = {names_path}\n')
    f.write(f'backup = {os.path.join(DARKNET_DIR, "backup")}\n')
print(f'作成: {data_path}')

# ----- 確認 -----
print('\n=== obj.data 内容 ===')
with open(data_path) as f:
    print(f.read())

# ラベルの整合性確認 (先頭5ファイル)
print('=== ラベル整合性確認 (先頭5件) ===')
train_list = os.path.join(DATA_DIR, 'train.txt')
with open(train_list) as f:
    lines = f.readlines()[:5]
for line in lines:
    img_path = line.strip()
    # darknetは画像パスの拡張子を.txtに置換してラベルを探す
    lbl_path = os.path.splitext(img_path)[0] + '.txt'
    img_ok = os.path.exists(img_path)
    lbl_ok = os.path.exists(lbl_path)
    print(f'  img={img_ok} lbl={lbl_ok} {os.path.basename(img_path)}')

---
## Step 3: YOLO-Fastest cfgファイル作成

YOLO-Fastest V1 のcfgを人物検出用にカスタマイズします。

### 主な変更点
- 入力: 192x192x3 (RGB)
- classes: 1 (person)
- filters: (1+5) x 3 = 18 (各[yolo]レイヤー前の[convolutional])
- batch/subdivisions: GPUメモリに応じて調整
- max_batches: 10000 (1クラス x 2000 x 5)

In [ ]:
import os

DARKNET_DIR = '/content/Yolo-Fastest'
CFG_DIR = os.path.join(DARKNET_DIR, 'cfg')
os.makedirs(CFG_DIR, exist_ok=True)

# Yolo-Fastestリポジトリ内のcfgファイルを確認
print('=== リポジトリ内のcfgファイル ===')
for root, dirs, files in os.walk(DARKNET_DIR):
    for f in files:
        if f.endswith('.cfg'):
            print(f'  {os.path.relpath(os.path.join(root, f), DARKNET_DIR)}')

In [ ]:
import os

DARKNET_DIR = '/content/Yolo-Fastest'
CFG_DIR = os.path.join(DARKNET_DIR, 'cfg')
os.makedirs(CFG_DIR, exist_ok=True)

# YOLO-Fastest V1 person検出用cfg (192x192x3 RGB)
# yolo-fastest-1.1.cfg または yolo-fastest-1.1_body.cfg をベースに作成
# アンカーはデフォルト値。Step 4でデータセットに合わせて再計算する

CFG_PATH = os.path.join(CFG_DIR, 'yolo-fastest-person-192.cfg')

cfg_content = """[net]
batch=64
subdivisions=16
width=192
height=192
channels=3
momentum=0.9
decay=0.0005
angle=0
saturation=1.5
exposure=1.5
hue=.1

learning_rate=0.001
burn_in=1000
max_batches=10000
policy=steps
steps=8000,9000
scales=.1,.1

# 0 - Backbone: Shufflenet v2 0.5x
[convolutional]
batch_normalize=1
filters=16
size=3
stride=2
pad=1
activation=leaky

[maxpool]
size=2
stride=2

# 2 - Channel split
[convolutional]
batch_normalize=1
filters=8
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=8
size=3
stride=1
pad=1
groups=8
activation=linear

[convolutional]
batch_normalize=1
filters=8
size=1
stride=1
pad=1
activation=leaky

# 5 - Route and shuffle
[route]
layers=-1,-4

[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

# 7 - Stride 2 block
[convolutional]
batch_normalize=1
filters=16
size=3
stride=2
pad=1
groups=16
activation=linear

[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-3

[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=16
size=3
stride=2
pad=1
groups=16
activation=linear

[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

# 14 - Route
[route]
layers=-1,-5

[convolutional]
batch_normalize=1
filters=32
size=1
stride=1
pad=1
activation=leaky

# 16 - Repeat block
[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=16
size=3
stride=1
pad=1
groups=16
activation=linear

[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-4

[convolutional]
batch_normalize=1
filters=32
size=1
stride=1
pad=1
activation=leaky

# 22 - Stride 2 block (to 12x12)
[convolutional]
batch_normalize=1
filters=32
size=3
stride=2
pad=1
groups=32
activation=linear

[convolutional]
batch_normalize=1
filters=24
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-3

[convolutional]
batch_normalize=1
filters=24
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=24
size=3
stride=2
pad=1
groups=24
activation=linear

[convolutional]
batch_normalize=1
filters=24
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-5

[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

# 30 - Repeat blocks
[convolutional]
batch_normalize=1
filters=24
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=24
size=3
stride=1
pad=1
groups=24
activation=linear

[convolutional]
batch_normalize=1
filters=24
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-4

[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

# 35 - Stride 2 block (to 6x6)
[convolutional]
batch_normalize=1
filters=48
size=3
stride=2
pad=1
groups=48
activation=linear

[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-3

[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=48
size=3
stride=2
pad=1
groups=48
activation=linear

[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-5

[convolutional]
batch_normalize=1
filters=96
size=1
stride=1
pad=1
activation=leaky

# 43 - Repeat block
[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=48
size=3
stride=1
pad=1
groups=48
activation=linear

[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-4

[convolutional]
batch_normalize=1
filters=96
size=1
stride=1
pad=1
activation=leaky

######## YOLO Head ########

# 48 - Detection head branch 0 (6x6, stride 32)
[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[convolutional]
size=1
stride=1
pad=1
filters=18
activation=linear

[yolo]
mask=3,4,5
anchors=10,14, 23,27, 37,58, 81,82, 135,169, 344,319
classes=1
num=6
jitter=.3
ignore_thresh=.7
truth_thresh=1
random=0
scale_x_y=1.05

# 51 - Upsample and route for branch 1 (12x12, stride 16)
[route]
layers=-3

[upsample]
stride=2

# Route to 12x12 feature map from backbone
[route]
layers=-1,29

# 54 - Detection head branch 1 (12x12, stride 16)
[convolutional]
batch_normalize=1
filters=48
size=1
stride=1
pad=1
activation=leaky

[convolutional]
size=1
stride=1
pad=1
filters=18
activation=linear

[yolo]
mask=0,1,2
anchors=10,14, 23,27, 37,58, 81,82, 135,169, 344,319
classes=1
num=6
jitter=.3
ignore_thresh=.7
truth_thresh=1
random=0
scale_x_y=1.05
"""

with open(CFG_PATH, 'w') as f:
    f.write(cfg_content)

print(f'cfgファイル作成: {CFG_PATH}')
print()
print('=== 重要な設定値 ===')
print(f'  入力: 192x192x3 (RGB)')
print(f'  クラス数: 1 (person)')
print(f'  出力 filters: 18 = (1+5)*3')
print(f'  max_batches: 10000')
print(f'  batch: 64, subdivisions: 16')
print()
print('NOTE: アンカーはデフォルト値です。Step 4でデータセットに合わせて再計算します。')
print()
print('NOTE: このcfgは簡略化したテンプレートです。')
print('リポジトリ内に yolo-fastest-1.1.cfg がある場合はそちらをベースに修正することを推奨します。')
print('次のセルでリポジトリ内のcfgをベースにした版も作成します。')

In [ ]:
import os
import re

DARKNET_DIR = '/content/Yolo-Fastest'
CFG_DIR = os.path.join(DARKNET_DIR, 'cfg')

# リポジトリ内の既存cfgをベースに人物検出用に修正する版
# yolo-fastest-1.1.cfg または yolo-fastest-1.1_body.cfg を探す
base_cfg = None
for candidate in [
    os.path.join(DARKNET_DIR, 'ModelZoo', 'yolo-fastest-1.1_body', 'yolo-fastest-1.1_body.cfg'),
    os.path.join(DARKNET_DIR, 'ModelZoo', 'yolo-fastest-1.1_coco', 'yolo-fastest-1.1.cfg'),
    os.path.join(DARKNET_DIR, 'cfg', 'yolo-fastest-1.1.cfg'),
    os.path.join(DARKNET_DIR, 'yolo-fastest-1.1.cfg'),
]:
    if os.path.exists(candidate):
        base_cfg = candidate
        break

if base_cfg:
    print(f'ベースcfgが見つかりました: {base_cfg}')
    with open(base_cfg) as f:
        content = f.read()

    # 入力サイズを192x192に変更
    content = re.sub(r'width=\d+', 'width=192', content)
    content = re.sub(r'height=\d+', 'height=192', content)

    # channels=3 (RGB)
    content = re.sub(r'channels=\d+', 'channels=3', content)

    # classes=1
    content = re.sub(r'classes=\d+', 'classes=1', content)

    # max_batches調整 (約50エポック相当 (スクラッチ学習に十分な反復回数))
    content = re.sub(r'max_batches\s*=\s*\d+', 'max_batches=50000', content)
    content = re.sub(r'steps\s*=\s*[\d,]+', 'steps=40000,45000', content)

    # [yolo]レイヤー前の[convolutional]のfiltersを修正
    # filters = (classes + 5) * num_anchors_per_branch = (1+5)*3 = 18
    # この置換は複雑なので、手動確認を推奨
    lines = content.split('\n')
    new_lines = []
    i = 0
    while i < len(lines):
        # [yolo]セクションの前の[convolutional]を見つけてfiltersを修正
        if i < len(lines) - 1 and lines[i].strip() == '[yolo]':
            # 前のセクションを遍り、直前の[convolutional]のfiltersを修正
            for j in range(len(new_lines) - 1, -1, -1):
                if 'filters=' in new_lines[j] and not new_lines[j].strip().startswith('#'):
                    new_lines[j] = 'filters=18'
                    break
                if new_lines[j].strip().startswith('[') and new_lines[j].strip() != '[convolutional]':
                    break
        new_lines.append(lines[i])
        i += 1

    modified_content = '\n'.join(new_lines)

    modified_cfg_path = os.path.join(CFG_DIR, 'yolo-fastest-person-192.cfg')
    with open(modified_cfg_path, 'w') as f:
        f.write(modified_content)
    print(f'\n修正版cfg作成: {modified_cfg_path}')
    print('\n=== 確認: [yolo]セクションの設定 ===')
    # [yolo]セクションを抽出して表示
    in_yolo = False
    yolo_count = 0
    for line in modified_content.split('\n'):
        if line.strip() == '[yolo]':
            in_yolo = True
            yolo_count += 1
            print(f'\n--- [yolo] #{yolo_count} ---')
        elif line.strip().startswith('['):
            in_yolo = False
        if in_yolo and ('classes' in line or 'mask' in line or 'anchors' in line or 'num' in line):
            print(f'  {line.strip()}')
else:
    print('WARNING: リポジトリ内にベースcfgが見つかりませんでした。')
    print('前のセルで作成したテンプレートcfgを使用します。')
    print('リポジトリ内のcfgファイルを確認し、必要に応じて手動修正してください。')


---
## Step 4: アンカー計算

データセットのバウンディングボックス分布に最適化したアンカーを計算します。

計算後、cfgファイルのanchors行を更新してください。

In [ ]:
%%bash
cd /content/Yolo-Fastest

# darknet calc_anchors で最適アンカーを計算
# 6アンカー (2ブランチ x 3アンカー)
./darknet detector calc_anchors \
    data/person/obj.data \
    -num_of_clusters 6 \
    -width 192 -height 192 \
    -show 2>&1 || echo 'NOTE: calc_anchorsが失敗した場合、以下のPythonセルで計算します'

In [ ]:
# Pythonでアンカー計算 (darknet calc_anchorsのフォールバック)
import numpy as np
import os
import glob

DATASET_DIR = '/content/dataset'
IMG_SIZE = 192
NUM_CLUSTERS = 6

# 全ラベルからバウンディングボックスのw,hを収集
boxes = []
for split in ['train']:
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    if not os.path.isdir(lbl_dir):
        continue
    for lbl_file in sorted(os.listdir(lbl_dir)):
        if not lbl_file.endswith('.txt'):
            continue
        with open(os.path.join(lbl_dir, lbl_file)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    w = float(parts[3]) * IMG_SIZE
                    h = float(parts[4]) * IMG_SIZE
                    if w > 0 and h > 0:
                        boxes.append([w, h])

boxes = np.array(boxes)
print(f'総ボックス数: {len(boxes)}')
print(f'W 統計: min={boxes[:,0].min():.1f}, max={boxes[:,0].max():.1f}, mean={boxes[:,0].mean():.1f}')
print(f'H 統計: min={boxes[:,1].min():.1f}, max={boxes[:,1].max():.1f}, mean={boxes[:,1].mean():.1f}')

# K-means clustering (IoUベース)
def iou_distance(box, clusters):
    """1 - IoU distance"""
    w = np.minimum(box[0], clusters[:, 0])
    h = np.minimum(box[1], clusters[:, 1])
    intersection = w * h
    box_area = box[0] * box[1]
    cluster_area = clusters[:, 0] * clusters[:, 1]
    union = box_area + cluster_area - intersection
    return 1.0 - intersection / union

def kmeans_anchors(boxes, k, max_iter=300):
    n = len(boxes)
    # 初期クラスタ中心をランダムに選択
    np.random.seed(42)
    indices = np.random.choice(n, k, replace=False)
    clusters = boxes[indices].copy()

    for _ in range(max_iter):
        distances = np.array([iou_distance(box, clusters) for box in boxes])
        assignments = np.argmin(distances, axis=1)

        new_clusters = np.zeros_like(clusters)
        for i in range(k):
            mask = assignments == i
            if mask.sum() > 0:
                new_clusters[i] = boxes[mask].mean(axis=0)
            else:
                new_clusters[i] = clusters[i]

        if np.allclose(clusters, new_clusters):
            break
        clusters = new_clusters

    # 面積でソート
    areas = clusters[:, 0] * clusters[:, 1]
    order = np.argsort(areas)
    return clusters[order]

anchors = kmeans_anchors(boxes, NUM_CLUSTERS)

print(f'\n=== 計算されたアンカー ({NUM_CLUSTERS}個) ===')
anchor_str_parts = []
for i, (w, h) in enumerate(anchors):
    print(f'  [{i}] w={w:.0f}, h={h:.0f} (area={w*h:.0f})')
    anchor_str_parts.append(f'{w:.0f},{h:.0f}')

anchor_str = ', '.join(anchor_str_parts)
print(f'\ndarknet cfg形式:')
print(f'anchors = {anchor_str}')
print(f'\nBranch 1 (12x12, stride 16): mask=0,1,2 -> {anchor_str_parts[0]}, {anchor_str_parts[1]}, {anchor_str_parts[2]}')
print(f'Branch 0 (6x6,  stride 32): mask=3,4,5 -> {anchor_str_parts[3]}, {anchor_str_parts[4]}, {anchor_str_parts[5]}')
print()
print('*** cfgファイルのanchors行を上記の値で更新してください ***')

In [ ]:
# cfgファイルのアンカーを更新
import re

CFG_PATH = '/content/Yolo-Fastest/cfg/yolo-fastest-person-192.cfg'

# anchor_str は前のセルで計算済み
try:
    with open(CFG_PATH) as f:
        cfg = f.read()

    # anchors行を置換
    cfg_updated = re.sub(r'anchors\s*=\s*[\d, .]+', f'anchors={anchor_str}', cfg)

    with open(CFG_PATH, 'w') as f:
        f.write(cfg_updated)

    print(f'cfg更新完了: {CFG_PATH}')
    print(f'anchors = {anchor_str}')
except NameError:
    print('ERROR: anchor_str が定義されていません。前のセルを先に実行してください。')

---
## Step 5: 事前学習済み重み取得

YOLO-Fastestのバックボーン重みを取得して転移学習のベースとします。

### 重みの取得方法
1. リポジトリ内のModelZooから事前学習済み重みを取得
2. `darknet partial` でバックボーン部分のみ抽出
3. 見つからない場合はスクラッチ学習

In [ ]:
import os
import glob

DARKNET_DIR = '/content/Yolo-Fastest'

# リポジトリ内の重みファイルを探す
print('=== リポジトリ内の重みファイル ===')
weights_files = glob.glob(os.path.join(DARKNET_DIR, '**/*.weights'), recursive=True)
for wf in weights_files:
    size_kb = os.path.getsize(wf) / 1024
    print(f'  {os.path.relpath(wf, DARKNET_DIR)}: {size_kb:.1f} KB')

if not weights_files:
    print('  重みファイルが見つかりません。ダウンロードを試みます。')

In [ ]:
%%bash
cd /content/Yolo-Fastest

# YOLO-Fastest 事前学習済み重みの取得を試みる
# ModelZoo内の重みまたはダウンロードリンクを確認

PRETRAINED=""

# パターン1: ModelZoo内の重み
if [ -f ModelZoo/yolo-fastest-1.1-xl.weights ]; then
    PRETRAINED="ModelZoo/yolo-fastest-1.1-xl.weights"
elif [ -f ModelZoo/yolo-fastest-1.1.weights ]; then
    PRETRAINED="ModelZoo/yolo-fastest-1.1.weights"
fi

# パターン2: リリースページからダウンロード
if [ -z "$PRETRAINED" ]; then
    echo 'リポジトリ内に重みが見つかりません。'
    echo 'COCO事前学習済み重みのダウンロードを試みます...'

    # dog-qiuqiuのGoogle Driveからダウンロード (URLは変更される可能性あり)
    # README.md内のリンクを確認
    echo ''
    echo 'NOTE: 事前学習済み重みが取得できない場合はスクラッチ学習を行います。'
    echo 'スクラッチ学習でも十分な精度が得られる場合があります。'
fi

if [ -n "$PRETRAINED" ]; then
    echo "=== 事前学習済み重み: $PRETRAINED ==="
    ls -la $PRETRAINED

    # バックボーン部分のみ抽出 (cfgのレイヤー数を確認して調整)
    echo '\nバックボーン重みの抽出を試みます...'
    ./darknet partial cfg/yolo-fastest-person-192.cfg $PRETRAINED yolo-fastest-backbone.conv 47 2>&1 || \
        echo 'WARNING: partial抽出に失敗。フル重みを使用します。'
else
    echo '=== スクラッチ学習を行います ==='
fi

---
## Step 6: モデル学習

darknet detector train で学習を実行します。

### 接続切れからの再開
学習結果はGoogle Driveに定期的にコピーされます。
接続が切れた場合:
1. ランタイムを再起動
2. Step 1からStep 3まで再実行
3. Step 6の「学習再開」セルを実行

### 学習パラメータ
- max_batches: 10000 (1クラス x 2000 x 5)
- batch: 64, subdivisions: 16 (実効ミニバッチ=4)
- learning_rate: 0.001
- steps: 8000, 9000 (80%, 90%にてLR x 0.1)

In [ ]:
import os
import shutil
import glob

DARKNET_DIR = '/content/Yolo-Fastest'
BACKUP_DIR = os.path.join(DARKNET_DIR, 'backup')
GDRIVE_BACKUP = '/content/drive/MyDrive/yolo_fastest_darknet_person'
CFG_PATH = os.path.join(DARKNET_DIR, 'cfg', 'yolo-fastest-person-192.cfg')
DATA_PATH = os.path.join(DARKNET_DIR, 'data', 'person', 'obj.data')

os.makedirs(BACKUP_DIR, exist_ok=True)
os.makedirs(GDRIVE_BACKUP, exist_ok=True)
os.makedirs(os.path.join(GDRIVE_BACKUP, 'backup'), exist_ok=True)

# backupディレクトリをGoogle Driveへのシンボリックリンクに置換
# darknetが保存する重みが自動的にGoogle Driveに書き込まれる (Colab切断対策)
gdrive_backup_dir = os.path.join(GDRIVE_BACKUP, 'backup')
if os.path.islink(BACKUP_DIR):
    print(f'backupディレクトリは既にGoogle Driveへのシンボリックリンクです')
elif os.path.isdir(BACKUP_DIR):
    # 既存のbackup内の重みをGoogle Driveにコピーしてから置換
    import glob as _g
    for w in _g.glob(os.path.join(BACKUP_DIR, '*.weights')):
        dst = os.path.join(gdrive_backup_dir, os.path.basename(w))
        if not os.path.exists(dst):
            shutil.copy2(w, dst)
            print(f'  コピー: {os.path.basename(w)} -> Google Drive')
    shutil.rmtree(BACKUP_DIR)
    os.symlink(gdrive_backup_dir, BACKUP_DIR)
    print(f'backup -> Google Drive シンボリックリンク作成済み')
else:
    os.symlink(gdrive_backup_dir, BACKUP_DIR)
    print(f'backup -> Google Drive シンボリックリンク作成済み')

# Google Driveから前回の学習結果を復元 (再開時)
gdrive_last = os.path.join(GDRIVE_BACKUP, 'yolo-fastest-person-192_last.weights')
local_last = os.path.join(BACKUP_DIR, 'yolo-fastest-person-192_last.weights')

if os.path.exists(gdrive_last) and not os.path.exists(local_last):
    shutil.copy2(gdrive_last, local_last)
    print(f'Google Driveから復元: {gdrive_last}')
    print(f'  -> {local_last} ({os.path.getsize(local_last)/1024:.1f} KB)')

# 事前学習済み重みの決定
pretrained_weights = ''
backbone_weights = os.path.join(DARKNET_DIR, 'yolo-fastest-backbone.conv')
if os.path.exists(local_last):
    pretrained_weights = local_last
    print(f'\n前回の学習から再開: {pretrained_weights}')
elif os.path.exists(backbone_weights):
    pretrained_weights = backbone_weights
    print(f'\nバックボーン重みで転移学習: {pretrained_weights}')
else:
    print('\nスクラッチ学習を行います')

print(f'\ncfg: {CFG_PATH}')
print(f'data: {DATA_PATH}')
print(f'weights: {pretrained_weights if pretrained_weights else "(none - scratch)"}')


In [ ]:
%%bash
cd /content/Yolo-Fastest

CFG="cfg/yolo-fastest-person-192.cfg"
DATA="data/person/obj.data"

# 重みファイルの決定
if [ -f backup/yolo-fastest-person-192_last.weights ]; then
    WEIGHTS="backup/yolo-fastest-person-192_last.weights"
    echo "=== 前回の学習から再開 ==="
elif [ -f yolo-fastest-backbone.conv ]; then
    WEIGHTS="yolo-fastest-backbone.conv"
    echo "=== バックボーン重みで転移学習 ==="
else
    WEIGHTS=""
    echo "=== スクラッチ学習 ==="
fi

echo "cfg:     $CFG"
echo "data:    $DATA"
echo "weights: $WEIGHTS"
echo ""

# 学習実行
./darknet detector train $DATA $CFG $WEIGHTS \
    -dont_show -gpus 0 2>&1 | tee /content/training_log.txt


In [ ]:
# 学習結果をGoogle Driveにバックアップ
import shutil
import os
import glob

BACKUP_DIR = '/content/Yolo-Fastest/backup'
GDRIVE_BACKUP = '/content/drive/MyDrive/yolo_fastest_darknet_person'

print('=== Google Driveへバックアップ ===')

# 重みファイルをコピー
for wf in glob.glob(os.path.join(BACKUP_DIR, '*.weights')):
    dst = os.path.join(GDRIVE_BACKUP, os.path.basename(wf))
    shutil.copy2(wf, dst)
    size_kb = os.path.getsize(wf) / 1024
    print(f'  {os.path.basename(wf)}: {size_kb:.1f} KB')

# 学習ログもコピー
log_file = '/content/training_log.txt'
if os.path.exists(log_file):
    shutil.copy2(log_file, os.path.join(GDRIVE_BACKUP, 'training_log.txt'))
    print(f'  training_log.txt')

# cfgもコピー
cfg_path = '/content/Yolo-Fastest/cfg/yolo-fastest-person-192.cfg'
if os.path.exists(cfg_path):
    shutil.copy2(cfg_path, os.path.join(GDRIVE_BACKUP, 'yolo-fastest-person-192.cfg'))
    print(f'  yolo-fastest-person-192.cfg')

# chart.png (darknetが生成する学習曲線)
chart_png = '/content/Yolo-Fastest/chart.png'
if os.path.exists(chart_png):
    shutil.copy2(chart_png, os.path.join(GDRIVE_BACKUP, 'chart.png'))
    print(f'  chart.png')

print(f'\n保存先: {GDRIVE_BACKUP}')

---
## Step 7: 精度評価

In [ ]:
%%bash
cd /content/Yolo-Fastest

CFG="cfg/yolo-fastest-person-192.cfg"
DATA="data/person/obj.data"

# best weightsを使用
BEST="backup/yolo-fastest-person-192_best.weights"
if [ ! -f "$BEST" ]; then
    # Google Driveから復元
    GDRIVE_BEST="/content/drive/MyDrive/yolo_fastest_darknet_person/yolo-fastest-person-192_best.weights"
    if [ -f "$GDRIVE_BEST" ]; then
        cp "$GDRIVE_BEST" "$BEST"
        echo "Google Driveからbest weightsを復元しました"
    else
        # final weightsを試す
        BEST="backup/yolo-fastest-person-192_final.weights"
    fi
fi

if [ -f "$BEST" ]; then
    echo "=== 精度評価: $BEST ==="
    echo ""

    # 評価用にcfgをテストモードに変更
    cp $CFG cfg/yolo-fastest-person-192-test.cfg
    sed -i 's/batch=64/batch=1/' cfg/yolo-fastest-person-192-test.cfg
    sed -i 's/subdivisions=16/subdivisions=1/' cfg/yolo-fastest-person-192-test.cfg

    ./darknet detector map $DATA cfg/yolo-fastest-person-192-test.cfg $BEST
else
    echo "ERROR: best weightsが見つかりません"
    echo "学習 (Step 6) を先に実行してください"
fi

---
## Step 8: darknet weights -> TFLite INT8 変換

これが最も重要なステップです。

### 変換パス
```
darknet .weights -> Keras .h5 -> TFLite FP32 -> TFLite INT8
```

### 使用ツール
- [david8862/keras-YOLOv3-model-set](https://github.com/david8862/keras-YOLOv3-model-set)

このツールはYOLOv3系のcfg/weightsをKerasに変換する機能を提供しています。

In [ ]:
# keras-YOLOv3-model-set のセットアップ
import os

CONVERTER_DIR = '/content/keras-YOLOv3-model-set'

if not os.path.isdir(CONVERTER_DIR):
    !git clone https://github.com/david8862/keras-YOLOv3-model-set.git {CONVERTER_DIR}
else:
    print(f'{CONVERTER_DIR} は既に存在します')

# 依存パッケージ
!pip install -q tensorflow keras matplotlib pillow tf-keras

# NumPy 2.0互換性修正: np.product -> np.prod
import subprocess
result = subprocess.run(['grep', '-rl', 'np.product', CONVERTER_DIR], capture_output=True, text=True)
if result.stdout.strip():
    !grep -rl 'np.product' {CONVERTER_DIR} | xargs sed -i 's/np.product/np.prod/g'
    print('np.product -> np.prod 修正済み')

# Keras 3.x互換性修正: tf-keras (Keras 2) を使用
convert_py = os.path.join(CONVERTER_DIR, 'tools', 'model_converter', 'convert.py')
with open(convert_py, 'r') as f:
    content = f.read()
content = content.replace('from tensorflow.keras', 'from tf_keras')
content = content.replace('from keras.', 'from tf_keras.')
content = content.replace('import keras', 'import tf_keras as keras')
with open(convert_py, 'w') as f:
    f.write(content)
print('convert.py を tf-keras 対応に修正済み')

print('=== セットアップ完了 ===')


In [ ]:
import os

DARKNET_DIR = '/content/Yolo-Fastest'
CONVERTER_DIR = '/content/keras-YOLOv3-model-set'
CFG_PATH = os.path.join(DARKNET_DIR, 'cfg', 'yolo-fastest-person-192.cfg')
BEST_WEIGHTS = os.path.join(DARKNET_DIR, 'backup', 'yolo-fastest-person-192_final.weights')
KERAS_H5_PATH = '/content/yolo_fastest_person.h5'

# Google Driveから復元 (必要に応じて)
if not os.path.exists(BEST_WEIGHTS):
    gdrive_best = '/content/drive/MyDrive/yolo_fastest_darknet_person/yolo-fastest-person-192_final.weights'
    if os.path.exists(gdrive_best):
        import shutil
        os.makedirs(os.path.dirname(BEST_WEIGHTS), exist_ok=True)
        shutil.copy2(gdrive_best, BEST_WEIGHTS)
        print(f'Google Driveから復元: {BEST_WEIGHTS}')

if not os.path.exists(BEST_WEIGHTS):
    print(f'ERROR: {BEST_WEIGHTS} が見つかりません')
    print('学習 (Step 6) を先に実行してください')
else:
    print(f'cfg: {CFG_PATH}')
    print(f'weights: {BEST_WEIGHTS} ({os.path.getsize(BEST_WEIGHTS)/1024:.1f} KB)')

    # Step 1: darknet -> Keras .h5 変換
    print('\n=== Step 8-1: darknet -> Keras .h5 ===')
    os.chdir(CONVERTER_DIR)
    !python tools/model_converter/convert.py \
        {CFG_PATH} \
        {BEST_WEIGHTS} \
        {KERAS_H5_PATH}

    if os.path.exists(KERAS_H5_PATH):
        print(f'\nKerasモデル保存: {KERAS_H5_PATH} ({os.path.getsize(KERAS_H5_PATH)/1024:.1f} KB)')
    else:
        print('ERROR: Kerasモデルの変換に失敗しました')
        print('\ncfgファイルが変換ツールと互換性がない可能性があります。')
        print('代替方法: 以下のセルで別の変換ツールを試します。')


In [ ]:
# 代替変換方法: Lebhoryi/yolo-fastest_inference を使用
# david8862で失敗した場合のみ実行
import os

KERAS_H5_PATH = '/content/yolo_fastest_person.h5'

if not os.path.exists(KERAS_H5_PATH):
    print('=== 代替変換: Lebhoryi/yolo-fastest_inference ===')

    ALT_DIR = '/content/yolo-fastest_inference'
    if not os.path.isdir(ALT_DIR):
        !git clone https://github.com/Lebhoryi/yolo-fastest_inference.git {ALT_DIR}

    os.chdir(ALT_DIR)
    !pip install -q configparser

    CFG_PATH = '/content/Yolo-Fastest/cfg/yolo-fastest-person-192.cfg'
    BEST_WEIGHTS = '/content/Yolo-Fastest/backup/yolo-fastest-person-192_best.weights'

    # このツールの変換スクリプトを確認
    !ls -la *.py convert/ 2>/dev/null || echo 'scriptsを確認中...'
    !find . -name '*.py' -type f | head -20

    print('\n上記のスクリプトを確認し、適切な変換コマンドを実行してください。')
else:
    print(f'Kerasモデルが既に存在します: {KERAS_H5_PATH}')
    print('このセルはスキップします。')

In [ ]:
import os
import numpy as np
import glob
from PIL import Image
import tf_keras
import tensorflow as tf

KERAS_H5_PATH = '/content/yolo_fastest_person.h5'
FP32_PATH = '/content/yolo_fastest_person_fp32.tflite'
INT8_PATH = '/content/yolo_fastest_person_darknet_int8.tflite'
DATASET_DIR = '/content/dataset'
IMG_SIZE = 192

if not os.path.exists(KERAS_H5_PATH):
    print('ERROR: Kerasモデルが見つかりません。Step 8-1を実行してください。')
else:
    # tf-keras (Keras 2) でモデルを読み込み
    print('=== Step 8-2: Keras -> TFLite FP32 ===')
    model = tf_keras.models.load_model(KERAS_H5_PATH, compile=False)

    # 動的入力形状を192x192x3に固定 (TFLite変換に必要)
    fixed_input = tf_keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image_input')
    outputs = model(fixed_input)
    fixed_model = tf_keras.Model(inputs=fixed_input, outputs=outputs)
    fixed_model.summary(line_length=120)

    # FP32 TFLite
    converter = tf.lite.TFLiteConverter.from_keras_model(fixed_model)
    tflite_fp32 = converter.convert()
    with open(FP32_PATH, 'wb') as f:
        f.write(tflite_fp32)
    print(f'\nFP32 TFLite: {os.path.getsize(FP32_PATH)/1024:.1f} KB')

    # INT8量子化
    print('\n=== Step 8-3: TFLite INT8 量子化 ===')
    cal_dir = os.path.join(DATASET_DIR, 'images', 'val')
    cal_images = sorted(glob.glob(os.path.join(cal_dir, '*.jpg')) +
                        glob.glob(os.path.join(cal_dir, '*.png')))[:200]
    print(f'キャリブレーション画像: {len(cal_images)}枚')

    def representative_dataset():
        for img_path in cal_images:
            img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
            arr = np.array(img, dtype=np.float32) / 255.0
            arr = arr.reshape(1, IMG_SIZE, IMG_SIZE, 3)
            yield [arr]

    converter = tf.lite.TFLiteConverter.from_keras_model(fixed_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    try:
        int8_model = converter.convert()
        with open(INT8_PATH, 'wb') as f:
            f.write(int8_model)
        int8_kb = os.path.getsize(INT8_PATH) / 1024
        print(f'\nINT8 TFLite: {int8_kb:.1f} KB')
    except Exception as e:
        print(f'INT8 量子化エラー: {e}')

    # 入出力テンソル確認
    print('\n--- 入出力テンソル確認 ---')
    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()
    print('入力:')
    for d in interp.get_input_details():
        qp = d.get('quantization_parameters', {})
        print(f'  {d["name"]}: shape={d["shape"]}, scale={qp["scales"][0]:.8f}, zp={qp["zero_points"][0]}')
    print('出力:')
    for d in interp.get_output_details():
        qp = d.get('quantization_parameters', {})
        print(f'  {d["name"]}: shape={d["shape"]}, scale={qp["scales"][0]:.8f}, zp={qp["zero_points"][0]}')


---
## Step 9: モデル詳細確認

INT8 TFLiteモデルの入出力詳細を確認します。

**重要**: 出力テンソルのscale/zero_pointはMCU側の後処理コードに必要です。必ず記録してください。

In [ ]:
import tensorflow as tf
import numpy as np
import os

INT8_PATH = '/content/yolo_fastest_person_darknet_int8.tflite'

if not os.path.exists(INT8_PATH):
    print(f'ERROR: {INT8_PATH} が見つかりません。Step 8を実行してください。')
else:
    file_size_kb = os.path.getsize(INT8_PATH) / 1024
    print('=' * 60)
    print('INT8 TFLite モデル詳細')
    print('=' * 60)
    print(f'ファイル: {INT8_PATH}')
    print(f'サイズ: {file_size_kb:.1f} KB ({file_size_kb/1024:.2f} MB)')
    print(f'NPUアリーナ制約 (432KB): {"OK" if file_size_kb <= 432 else "OVER"}')
    print()

    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    # 入力詳細
    print('--- 入力テンソル ---')
    for i, d in enumerate(interp.get_input_details()):
        print(f'  [{i}] name: {d["name"]}')
        print(f'       shape: {d["shape"]}')
        print(f'       dtype: {d["dtype"]}')
        qp = d.get('quantization_parameters', {})
        sc = qp.get('scales', np.array([]))
        zp = qp.get('zero_points', np.array([]))
        if len(sc) > 0:
            print(f'       scale: {sc[0]:.8f}')
            print(f'       zero_point: {zp[0]}')
    print()

    # 出力詳細
    print('--- 出力テンソル ---')
    for i, d in enumerate(interp.get_output_details()):
        print(f'  [{i}] name: {d["name"]}')
        print(f'       shape: {d["shape"]}')
        print(f'       dtype: {d["dtype"]}')
        qp = d.get('quantization_parameters', {})
        sc = qp.get('scales', np.array([]))
        zp = qp.get('zero_points', np.array([]))
        if len(sc) > 0:
            print(f'       scale: {sc[0]:.8f}')
            print(f'       zero_point: {zp[0]}')
    print()

    # オペレータ一覧
    print('--- オペレータ一覧 ---')
    ops = set()
    tensor_details = interp.get_tensor_details()
    print(f'  テンソル数: {len(tensor_details)}')
    # TFLiteのオペレータ名は直接取得できないが、テンソル名から推定
    for td in tensor_details:
        name = td['name']
        for op_name in ['Conv2D', 'DepthwiseConv2D', 'MaxPool', 'Concatenate',
                        'Upsample', 'Reshape', 'LeakyRelu', 'Add', 'Mul',
                        'Sigmoid', 'StridedSlice', 'Pad', 'Resize']:
            if op_name.lower() in name.lower():
                ops.add(op_name)
    if ops:
        print(f'  検出されたオペレータ: {", ".join(sorted(ops))}')

    # Ethos-U55未対応オペレータの確認
    unsupported = ops & {'StridedSlice', 'Resize'}
    if unsupported:
        print(f'\n  WARNING: Ethos-U55未対応の可能性があるオペレータ: {unsupported}')
    else:
        print(f'\n  OK: Ethos-U55未対応オペレータは検出されませんでした')

    print()
    print('=' * 60)
    print('*** 重要: 上記の出力テンソルの scale / zero_point を控えてください ***')
    print('MCU側の後処理コード (DetectorPostProcessing.cc) に設定が必要です')
    print('=' * 60)

---
## Step 10: Vela互換性確認

Arm VelaコンパイラでEthos-U55との互換性を確認します。

確認ポイント:
- NPUに配置されるオペレータの割合 (100%が理想)
- CPUフォールバックオペレータの有無
- サブグラフ数 (1が理想)

In [ ]:
# Arm Vela コンパイラのインストールと確認
!pip install -q ethos-u-vela

print('=== Velaバージョン ===')
!vela --version 2>/dev/null || echo 'Velaのインストールに失敗しました'

In [ ]:
%%bash
INT8_PATH="/content/yolo_fastest_person_darknet_int8.tflite"
VELA_OUTPUT="/content/vela_output"

if [ ! -f "$INT8_PATH" ]; then
    echo "ERROR: INT8モデルが見つかりません。Step 8を実行してください。"
    exit 0
fi

mkdir -p $VELA_OUTPUT

echo "=== Velaコンパイル (Ethos-U55-256) ==="
echo ""

vela \
    --accelerator-config ethos-u55-256 \
    --system-config Ethos_U55_High_End_Embedded \
    --memory-mode Sram_Only \
    --output-dir $VELA_OUTPUT \
    $INT8_PATH 2>&1

echo ""
echo "=== Vela出力ファイル ==="
ls -la $VELA_OUTPUT/ 2>/dev/null

# Vela最適化済みモデルのサイズ確認
VELA_MODEL=$(find $VELA_OUTPUT -name '*.tflite' | head -1)
if [ -n "$VELA_MODEL" ]; then
    SIZE_KB=$(du -k "$VELA_MODEL" | cut -f1)
    echo ""
    echo "Vela最適化後サイズ: ${SIZE_KB} KB"
    if [ $SIZE_KB -le 432 ]; then
        echo "OK: NPUアリーナ制約 (432KB) 以内"
    else
        echo "WARNING: NPUアリーナ制約 (432KB) 超過"
    fi
fi

---
## Step 11: 成果物ダウンロード

In [ ]:
# Google Drive に全成果物を保存
import shutil
import os
import json

OUTPUT_DIR = '/content/drive/MyDrive/yolo_fastest_darknet_person/model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('=== 成果物をGoogle Driveに保存 ===')

# コピー対象ファイル
files_to_copy = {
    '/content/Yolo-Fastest/backup/yolo-fastest-person-192_best.weights': 'yolo-fastest-person-192_best.weights',
    '/content/yolo_fastest_person.h5': 'yolo_fastest_person.h5',
    '/content/yolo_fastest_person_fp32.tflite': 'yolo_fastest_person_fp32.tflite',
    '/content/yolo_fastest_person_darknet_int8.tflite': 'yolo_fastest_person_darknet_int8.tflite',
    '/content/Yolo-Fastest/cfg/yolo-fastest-person-192.cfg': 'yolo-fastest-person-192.cfg',
    '/content/Yolo-Fastest/chart.png': 'training_chart.png',
    '/content/training_log.txt': 'training_log.txt',
}

# Vela出力
import glob
vela_models = glob.glob('/content/vela_output/*.tflite')
for vm in vela_models:
    files_to_copy[vm] = f'vela_{os.path.basename(vm)}'

for src, dst_name in files_to_copy.items():
    if os.path.exists(src):
        dst = os.path.join(OUTPUT_DIR, dst_name)
        shutil.copy2(src, dst)
        size_kb = os.path.getsize(src) / 1024
        print(f'  {dst_name}: {size_kb:.1f} KB')
    else:
        print(f'  SKIP: {dst_name} (ファイルなし)')

# 量子化パラメータをJSONで保存
INT8_PATH = '/content/yolo_fastest_person_darknet_int8.tflite'
if os.path.exists(INT8_PATH):
    import tensorflow as tf
    import numpy as np

    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    quant_info = {'input': [], 'output': []}
    for tag, details in [('input', interp.get_input_details()),
                         ('output', interp.get_output_details())]:
        for d in details:
            qp = d.get('quantization_parameters', {})
            sc = qp.get('scales', np.array([]))
            zp = qp.get('zero_points', np.array([]))
            info = {
                'name': d['name'],
                'shape': d['shape'].tolist(),
                'dtype': str(d['dtype']),
                'scale': float(sc[0]) if len(sc) > 0 else None,
                'zero_point': int(zp[0]) if len(zp) > 0 else None,
            }
            quant_info[tag].append(info)

    quant_json_path = os.path.join(OUTPUT_DIR, 'quantization_params.json')
    with open(quant_json_path, 'w') as f:
        json.dump(quant_info, f, indent=2)
    print(f'  quantization_params.json')

print(f'\n=== 保存先: {OUTPUT_DIR} ===')

In [ ]:
# INT8 TFLite モデルをダウンロード
from google.colab import files
import os

INT8_PATH = '/content/yolo_fastest_person_darknet_int8.tflite'

if os.path.exists(INT8_PATH):
    files.download(INT8_PATH)
    print('INT8モデルのダウンロードを開始しました')
else:
    print('INT8モデルが見つかりません。Step 8を先に実行してください。')

---
## まとめ

### 変換パス
```
darknet .weights  -->  Keras .h5  -->  TFLite FP32  -->  TFLite INT8
   (dog-qiuqiu)    (david8862)       (TF Lite)        (PTQ, INT8)
```

### 次のステップ (実機デプロイ)

1. INT8 TFLiteモデルをダウンロード
2. RUHMI `mcu_deploy.py --ethos` でMERA変換
3. **単一サブグラフ (sub_0000のみ) であることを確認**
   - sub_0001, sub_0002 が生成されなければ成功
4. 後処理コードをYOLOv3アンカーベースに書き換え
5. e2studioビルド -> 実機書き込み -> 動作確認

### 後処理パラメータ (MCUコードに設定が必要)
- 入力: 192x192x3 RGB (INT8)
- 出力: 2ブランチ (6x6 stride-32 + 12x12 stride-16)
- アンカー: Step 4で計算した値
- 出力テンソルのscale/zero_point: Step 9の値
- デコード: YOLOv3スタイル
  - objectness = sigmoid((int8_val - zero_point) * scale)
  - bbox: sigmoid(tx)+cx, sigmoid(ty)+cy, exp(tw)*anchor_w, exp(th)*anchor_h
  - class_score = sigmoid(class_val) * objectness
  - NMS (IoU閾値 0.45, 信頼度閾値 0.5)

### 後処理リファレンス
- 顔検出サンプル: `reference_projects/ruhmi-framework-mcu/application_examples/face_detection/src/ai_application/DetectorPostProcessing.cc`

### トラブルシューティング

| 問題 | 対処 |
|---|---|
| darknetコンパイル失敗 | OpenCVバージョン確認、`OPENCV=0`で再コンパイル |
| Keras変換失敗 | cfg形式の互換性確認、Lebhoryi/yolo-fastest_inferenceを試す |
| INT8量子化エラー | FP32 TFLiteのオペレータ確認、キャリブレーション画像数を増やす |
| Velaコンパイル失敗 | 未対応オペレータの特定、cfgの活性化関数を確認 (LeakyReLU/ReLU6のみ使用) |
| モデルサイズ>432KB | フィルタ数削減、またはSDRAM配置を検討 |